In [9]:
import polars as pl

In [ ]:
raw_data = pl.read_csv(
    r"test_path",
    schema_overrides={
        "pending_earn_balance": pl.Float64,
        "pending_redeem_balance": pl.Float64,
        "cleared_balance": pl.Float64,
        "currency_id": pl.Float64,
        "points_earned": pl.Float64,
    },
)

raw_data

In [ ]:
clean_data = (
    raw_data.with_columns(
        (pl.col("cleared_balance") - pl.col("points_earned")).alias("new_balance")
    )
    .select(["contact_number", "new_balance"])
    .filter(pl.col("new_balance") > 0)
    .filter(pl.col("contact_number").str.len_chars() == 12)
)

clean_data.write_csv("./test_files/2_columns_2.csv", include_header=False)
clean_data


In [ ]:
chunksize = 1_000
reader = pl.read_csv_batched(r"./test_files/2_columns_2.csv", batch_size=chunksize)

part = 0
while True:
    batches = reader.next_batches(1)  # get up to 1 DataFrame of batch_size
    if not batches:
        break
    for df in batches:
        part += 1
        print(f"Writing part {part}, rows={df.shape[0]}")
        df.write_csv(f"./test_files/big_part_2_{part:03d}.csv", include_header=False)